In [96]:
pip install pandas numpy requests

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [97]:
import os
os.chdir("..")   # go up from Course Project to assignments
os.chdir("Final Course Project")
os.getcwd()
os.listdir()

['DataFinalFlightCode.py',
 'DataFinalFlightsWIP.py',
 'cancun_weather_final.xlsx',
 'finalFlightData.csv',
 'ProjectCodeWIP.ipynb',
 'cancun_hotel_prices.csv',
 'readme.txt',
 'GG, AJ, BA CS 210 Project Proposal.pdf',
 '.ipynb_checkpoints',
 'cancun_hotel_prices.ipynb',
 'FinalProjectCode.ipynb']

In [ ]:
import pandas as pd
import numpy as np
import datetime as dt
import seaborn as sns
import matplotlib.pyplot as plt

path = "cancun_weather_final.xlsx"

# Working on parsing cancun_weatherr.xlsx into a Dataframe
# Using the row that contains 'time' as the header.
# This is row 4 (0-based index 3), but we can start with header=3 and adjust if needed.
importedWeather = pd.read_excel("cancun_weather_final.xlsx", header=3)
weather = importedWeather.dropna(subset=["time"]).copy()
weather["time"] = pd.to_datetime(weather["time"])
weather = weather.rename(columns={
    "time": "date",
    "uv_index_max ()": "uvIndexMax",
    "temperature_2m_max (Â°F)": "maxTemp",
    "temperature_2m_min (Â°F)": "minTemp",
    "precipitation_sum (mm)": "precipMM",
    "precipitation_hours (h)": "precipHrs",
    "cloud_cover_mean (%)": "avgCloudCover",
    "sunrise (iso8601)": "sunrise",
    "sunset (iso8601)": "sunset",
    "wind_speed_10m_max (mp/h)": "windSpeedMax",
    "wind_gusts_10m_max (mp/h)": "windGustsMax",
    "rain_sum (mm)": "rainSum",
    "showers_sum (mm)": "showersSum",
    "daylight_duration (s)": "daylightSeconds",
    "visibility_mean (m)": "visibilityMean",
    "precipitation_probability_max (%)": "precipProbMax",
})

# I need to make sure all of my columns are usable for a correlation matrix
weather["date"] = pd.to_datetime(weather["date"])
weather["sunrise"] = pd.to_datetime(weather["sunrise"])
weather["sunset"] = pd.to_datetime(weather["sunset"])

# Convert sunrise/sunset to seconds since midnight, as part of conversion to numerical values for correlation matrix
weather["sunrise"] = weather["sunrise"].dt.hour * 3600 + weather["sunrise"].dt.minute * 60
weather["sunset"] = weather["sunset"].dt.hour * 3600 + weather["sunset"].dt.minute * 60


# Building Master Date Range (2023 → 2026)
weather["doy"] = weather["date"].dt.dayofyear
masterDates = pd.date_range("2023-04-18", "2026-04-18")
master = pd.DataFrame({"date": masterDates})
master["doy"] = master["date"].dt.dayofyear


# Working on parsing flights_data.csv into a Dataframe
realFlyPrices = pd.read_csv("finalFlightData.csv")
realFlyPrices["date"] = pd.to_datetime(realFlyPrices["date"])
realFlyPrices["doy"] = realFlyPrices["date"].dt.dayofyear

baselineFlyPrices = realFlyPrices.groupby("doy")["price"].mean().reset_index()
baselineFlyPrices = baselineFlyPrices.rename(columns={"price": "baselinePrice"})

# Attach baseline curve to master curve
master = master.merge(baselineFlyPrices, on="doy", how="left")

# Fill all missing baseline values (interpolate + edge fill)
master["baselinePrice"] = (master["baselinePrice"].interpolate().ffill().bfill())
# Add noise
master["noisyPrices"] = master["baselinePrice"] * np.random.uniform(0.95, 1.05, size=len(master))

# Seasonal multipliers
def seasonal_and_holiday_multiplier(date):
    month = date.month
    day = date.day
    if month in [12, 1]: 
        return 1.15
    if month == 3 and 10 <= day <= 25: 
        return 1.30
    if month in [6,7,8]: 
        return 1.15
    if month in [9, 10]: 
        return 0.85
    return 1.0

master["flightPrices"] = master.apply(
    lambda row: row["noisyPrices"] * seasonal_and_holiday_multiplier(row["date"]),
    axis=1
)

# Now moving on to hotel prices. This is a much more complete dataset so I won't need to interpolate as much as each date has 3-5 data points informing it, as each hotel has prices fo
hotels = pd.read_csv("cancun_hotel_prices.csv")
hotels["date"] = pd.to_datetime(hotels["date"])
hotels["price"] = pd.to_numeric(hotels["price_usd"], errors="coerce")
hotels = hotels.dropna(subset=["price"])

# Averaging out all of the columns on the same date into a mean price
hotelDayAvgPrice = hotels.groupby("date")["price"].mean().reset_index()
hotelDayAvgPrice = hotelDayAvgPrice.rename(columns={"price": "hotelPrice"})
hotelDayAvgPrice["doy"] = hotelDayAvgPrice["date"].dt.dayofyear
# Attach hotel prices to master date range by date of year
master = master.merge(hotelDayAvgPrice[["doy", "hotelPrice"]], on="doy", how="left")

# Merge it all together
weather_latest = weather.sort_values("date").drop_duplicates("doy", keep="last")
master = master.merge(weather_latest.drop(columns=["date"]), on="doy", how="left")
master["hotelPrice"] = master["hotelPrice"].ffill().bfill()
merged = master.copy()

# print("Weather:", weather["date"].min(), weather["date"].max())
# print("Flights:", realFlyPrices["date"].min(), realFlyPrices["date"].max())
# print("Hotels:", hotelDayAvgPrice["date"].min(), hotelDayAvgPrice["date"].max())

# Correlation Matrix

correlationColumns = [
    "flightPrices",
    "hotelPrice",
    "uvIndexMax",
    "maxTemp",
    "minTemp",
    "precipMM",
    "precipHrs",
    "avgCloudCover",
    "sunrise",
    "sunset",
    "windSpeedMax",
    "windGustsMax",
    "rainSum",
    "showersSum",
    "daylightSeconds",
    "visibilityMean",
    "precipProbMax"
]
correlationMatrix = merged[correlationColumns].corr()

plt.figure(figsize=(14, 10))
sns.heatmap(correlationMatrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Weather, Flight Price, and Hotel Price Correlation Matrix")
plt.show()

flightCorr = correlationMatrix["flightPrices"]
hotelCorr  = correlationMatrix["hotelPrice"]
combinedCorrelation = (flightCorr + hotelCorr) / 2

for col in correlationColumns:
    if col not in ["flightPrices", "hotelPrice"]:
        merged[col + "Normalized"] = (merged[col] - merged[col].mean()) / merged[col].std()

def weather_multiplier(row, corr):
    multiplier = 1.0
    for col in corr.index:
        if col in ["flightPrices", "hotelPrice"]:
            continue
        normalizedCol = col + "Normalized"
        if normalizedCol in row:
            val = row[normalizedCol]
            corr_val = corr[col]
            # Skip if either side is NaN
            if pd.isna(val) or pd.isna(corr_val):
                continue
            multiplier += corr_val * val * 0.1
    return multiplier


merged["flightPrices"] = merged.apply(lambda row: row["flightPrices"] * weather_multiplier(row, combinedCorrelation), axis=1)
merged["hotelPrice"] = merged.apply(lambda row: row["hotelPrice"] * weather_multiplier(row, combinedCorrelation),axis=1)

absoluteCombinedCorr = (flightCorr.abs() + hotelCorr.abs()) / 2
dropThreshold = 0.20
strongCorrelations = absoluteCombinedCorr[absoluteCombinedCorr > dropThreshold].index.tolist()
weather_features = [
    col for col in strongCorrelations
    if col not in ["date", "flightPrices", "hotelPrice"]
]

def classify(row):
    if (row["precipMM"] > 7) & (row["precipHrs"] > 6):
        return "rainy"
    elif row["avgCloudCover"] > 84:
        return "cloudy"
    else:
        return "clear"

weather["genLabel"] = weather.apply(classify, axis=1)
merged[["date"] + strongCorrelations].head(100)


In [ ]:
from sklearn.ensemble import RandomForestRegressor
import matplotlib.pyplot as plt

df = merged[["date"] + strongCorrelations].copy()
df["date"] = pd.to_datetime(df["date"])
df["doy"] = df["date"].dt.dayofyear

# Build DOY weather averages
weatherAvg = df.groupby("doy")[strongCorrelations].mean().reset_index()

# ML inputs
X = df[strongCorrelations]
y = df[["flightPrices", "hotelPrice"]] 

model = RandomForestRegressor(
    n_estimators=300,
    random_state=42
)
model.fit(X, y)

# Build 2027 future dataset
resultDates = pd.date_range("2027-01-01", "2027-12-31")
future = pd.DataFrame({"date": resultDates})
future["date"] = pd.to_datetime(future["date"])
future["doy"] = future["date"].dt.dayofyear

# Merge weather averages
future = future.merge(weatherAvg, on="doy", how="left")
future["month"] = future["date"].dt.month
future["dayOfWeek"] = future["date"].dt.day_name()

# Predict 2027 prices
futureX = future[strongCorrelations]
predictions = model.predict(futureX)

future["flightPrices"] = predictions[:, 0]
future["hotelPrice"] = predictions[:, 1]

# Only drop weather features, NOT the predicted prices
weatherCols = [c for c in strongCorrelations if c not in ["flightPrices", "hotelPrice"]]
dropCols = weatherCols + ["doy"]

userFutures = future.drop(columns=dropCols)
userFutures["month"] = userFutures["date"].dt.strftime("%b")

print(userFutures.head(100))

plt.figure(figsize=(14, 6))

plt.plot(
    userFutures["date"],
    userFutures["flightPrices"],
    label="Flight Price",
    color="blue",
    linewidth=2
)

plt.plot(
    userFutures["date"],
    userFutures["hotelPrice"],
    label="Hotel Price",
    color="red",
    linewidth=2
)

plt.title("Predicted 2027 Flight & Hotel Prices")
plt.xlabel("Date")
plt.ylabel("Price (USD)")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [100]:
def searchTrip(future_df, arrival, departure):
    # Convert to datetime
    arrival = pd.to_datetime(arrival)
    departure = pd.to_datetime(departure)

    if departure <= arrival:
        raise ValueError("Departure date must be after arrival date")

    trip = future_df[(future_df["date"] >= arrival) & (future_df["date"] <= departure)].copy()

    if trip.empty:
        raise ValueError("No data available for the selected date range")

    # --- Flight prices ---
    # 
    oneWayPrice = trip.iloc[0]["flightPrices"]
    roundTripPrice = trip.iloc[0]["flightPrices"] + trip.iloc[-1]["flightPrices"]
    oneWayForTwo = oneWayPrice * 2
    roundTripForTwo = roundTripPrice * 2

    # --- Hotel prices ---
    numNights = (departure - arrival).days

    # Sum hotel prices for each night
    hotelTotal = trip.iloc[:-1]["hotelPrice"].sum()  # exclude departure day
    hotelPerNight = hotelTotal / numNights

    return {
        "arrival": arrival.date(),
        "departure": departure.date(),
        "nights": numNights,

        "One Way, One Person": round(oneWayPrice, 2),
        "Round Trip, One Person": round(roundTripPrice, 2),
        "One Way, Two People": round(oneWayForTwo, 2),
        "Round Trip, Two People": round(roundTripForTwo, 2),

        "Hotel Cost Total": round(hotelTotal, 2),
        "Hotel Cost Per Night": round(hotelPerNight, 2)
    }

def printSummary(results):
    arrival = results["arrival"].strftime("%B %d, %Y")
    departure = results["departure"].strftime("%B %d, %Y")
    
    print(f"Trip Summary")
    print(f"Arrival:   {arrival}")
    print(f"Departure: {departure}")
    print(f"Nights:    {results['nights']}\n")

    print("Flight Prices")
    print(f"• One-way (1 person):     ${results['One Way, One Person']:.2f}")
    print(f"• Round-trip (1 person):  ${results['Round Trip, One Person']:.2f}")
    print(f"• One-way (2 people):     ${results['One Way, Two People']:.2f}")
    print(f"• Round-trip (2 people):  ${results['Round Trip, Two People']:.2f}\n")

    print("Hotel Prices")
    print(f"• Total hotel cost:       ${results['Hotel Cost Total']:.2f}")
    print(f"• Price per night:        ${results['Hotel Cost Per Night']:.2f}")

printSummary(searchTrip(userFutures, "9-01-2027", "9-06-2027"))

Trip Summary
Arrival:   September 01, 2027
Departure: September 06, 2027
Nights:    5

Flight Prices
• One-way (1 person):     $181.25
• Round-trip (1 person):  $339.67
• One-way (2 people):     $362.50
• Round-trip (2 people):  $679.35

Hotel Prices
• Total hotel cost:       $1270.81
• Price per night:        $254.16


In [ ]:
# These functions were the functions that I used to generate synthetic hotel and flight prices, while assuming the adjustement for weather
# This was me assuming that certain weather patterns would be guaranteed to introduce variations in price and hard coding that in
# Upon recommendation from the TA's project help session, I decided to integrate our limited scrapped flight data into the codeset and only hard-code holiday multipliers in,
# while genertaing a correlation matrix on what weather factors impacted the price of flights and hotels more.
# I have the full dataset for hotels, so the only "factors" I am hard-coding to influence prices are seasonal multipliers on flight prices only
# Weather correlation variables will help me manually vary the rest of the flight prices, and help me determine which weather columns I will use in my machine learning dataset
"""
def weather_adjustment(row):
    temp_effect = (row["maxTemp"] - 80) * 0.003   # hotter days → slightly higher demand
    rain_effect = -0.04 * (row["precipMM"] - 7)           # heavy rain → lower demand
    cloud_effect = -0.005 * (row["avgCloudCover"] - 84)  # cloudier → slightly lower
    return temp_effect + rain_effect + cloud_effect
"""

"""
def generate_synthetic_prices(df):
    fromEWRFlight = 250   # typical Cancun price floor
    avg5StarHotel = 200    # typical Cancun hotel floor
    flights = []
    hotels = []
    
    for _, row in df.iterrows():
        date = row["date"]

        # Weather Adjustments
        season = seasonal_multiplier(date)
        w_adj = weather_adjustment(row)
        
        # Random noise (keeps it realistic)
        flightNoise = np.random.normal(-5, 12)
        hotelNoise = np.random.normal(-3, 9)
        
        # Final synthetic prices
        flightPrice = fromEWRFlight * season * (1 + w_adj) + flightNoise
        hotelPrice = avg5StarHotel * season * (1 + w_adj) + hotelNoise
        # Prevent unrealistic negatives
        flights.append(max(80, round(flightPrice, 2)))
        hotels.append(max(40, round(hotelPrice, 2)))
    
    df["flightPrice"] = flights
    df["hotelPrice"] = hotels
    return df
"""